[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/061.%20The%20Order%20Batching%20Problem/P61-Tier-4_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 61. The Order Batching Problem
## Tier 4: Multi-Agent Reinforcement Learning

### Purpose and scope
This notebook builds a small multi-agent order-batching simulation. Three agents repeatedly choose whether to wait, complete a batch, or add an available order to their batch. Each agent uses an Actor-Critic learning architecture with a neural-network policy and value function. The environment supplies shared information and rewards that include order-selection efficiency and coordination effects.

The inputs are a set of customer orders, a number of agents, and a capacity limit for each agent. The outputs are learned agent policies, episode-level training and evaluation metrics, and plots for interpreting those metrics. The example uses 50 orders, three agents, capacity 8, and the specified training and evaluation episode counts. Training is stochastic; results are produced by running the notebook, not assumed in advance.

### Modeling assumptions
- Each order has a volume, a two-dimensional location, a priority, and a deadline. Order locations are used to calculate Euclidean distance from a depot at \\( (0,0) \\).
- Agent capacity limits the total order volume in a batch. Agents do not travel through the warehouse: the distance-based reward uses each order's depot distance, and reported cumulative distance is a round-trip proxy, not a routed picking distance.
- An order can be assigned to at most one agent at a time. It becomes completed when its assigned agent completes the batch. Priority and remaining deadline are state features; this example does not give them separate reward terms.
- Agents choose actions from a shared state snapshot at each time step. Actions are then processed in agent order; if two agents choose the same order, the first valid selection is accepted and the later selection receives a conflict penalty.
- The small neural networks and replay-based updates are an educational implementation of Actor-Critic learning. Experience replay without importance-sampling correction is not an unbiased on-policy estimator, and the learned policy is not guaranteed to be globally optimal.

### How the method works
A **state** is the numerical information an agent uses to choose an action. An **action** is one available decision, such as selecting an order. A **reward** is a numerical signal from the environment that encourages or discourages that decision. Reinforcement learning uses repeated state-action-reward transitions to improve a policy, the rule that maps states to action preferences.

Each actor produces action logits, which are converted into probabilities after invalid actions are masked. Each critic estimates the value of a state. For agent \\( i \\), the one-step temporal-difference target and advantage are

$$
y_t^{(i)} = r_t^{(i)} + \\gamma (1-d_t) V_{\\phi_i}(s_{t+1}^{(i)}), \\qquad
A_t^{(i)} = y_t^{(i)} - V_{\\phi_i}(s_t^{(i)}).
$$

Here, \\(s_t^{(i)}\\) is agent \\(i\\)'s state, \\(r_t^{(i)}\\) is its reward, \\(d_t\\) is 1 at a terminal transition and 0 otherwise, \\(\\gamma\\) is the discount factor, and \\(V_{\\phi_i}\\) is the critic with parameters \\(\\phi_i\\). The target estimates immediate reward plus discounted future value. The advantage estimates whether the observed action outcome was better or worse than the critic expected.

For a policy \\(\\pi_{\\theta_i}\\), the learning signal is the policy-gradient loss

$$
L_{\\mathrm{actor}} = -\\mathbb{E}\\left[A_t^{(i)} \\log \\pi_{\\theta_i}(a_t^{(i)} \\mid s_t^{(i)})\\right],
\\qquad
L_{\\mathrm{critic}} = \\mathbb{E}\\left[(y_t^{(i)} - V_{\\phi_i}(s_t^{(i)}))^2\\right].
$$

The actor increases the probability of actions with positive advantage and decreases it for actions with negative advantage. The critic is trained to reduce the squared error between its value estimate and the temporal-difference target. The code implements these updates by backpropagating through the networks' layers.

### Training and interpretation
Training collects transitions in each agent's replay buffer and samples batches for updates. An epsilon-greedy rule adds exploration: with probability \\(\\epsilon\\), an agent chooses a random valid action; otherwise it samples from its masked policy. The training schedule gradually reduces \\(\\epsilon\\). Evaluation sets \\(\\epsilon=0\\) and reports observed rewards, completion rates, and completed-batch statistics. These are empirical results for the simulated instance, not guarantees about other warehouses or data.

The learning loop has a cost that depends on the number of episodes, steps per episode, agents, and network computation per update. Experience storage grows with the replay-buffer capacity and the size of each recorded transition. This example is intended to make the algorithmic flow inspectable; it is not a benchmark of production-scale training.

In [ ]:
from collections import defaultdict, deque
from dataclasses import dataclass
import random
import time
import warnings
from typing import List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Fixed seeds make the example reproducible when run with the same software stack.
np.random.seed(42)
random.seed(42)

print("=== Order Batching: Multi-Agent Reinforcement Learning ===")
print("Required packages imported successfully.")

In [ ]:
# A small fully connected network supplies the actor's logits or the critic's value estimate.
class SimpleNeuralNetwork:
    """Fully connected ReLU network with a linear output layer."""

    def __init__(self, input_dim, hidden_dims, output_dim):
        if input_dim <= 0 or output_dim <= 0:
            raise ValueError("Network input and output dimensions must be positive.")

        self.input_dim = input_dim
        self.hidden_dims = list(hidden_dims)
        self.output_dim = output_dim
        dimensions = [input_dim] + self.hidden_dims + [output_dim]
        self.weights = []
        self.biases = []

        for fan_in, fan_out in zip(dimensions[:-1], dimensions[1:]):
            # He initialization helps maintain activation scale through ReLU layers.
            self.weights.append(
                np.random.randn(fan_in, fan_out) * np.sqrt(2.0 / fan_in)
            )
            self.biases.append(np.zeros(fan_out))

    def forward(self, inputs):
        """Compute network outputs for one input or a batch of inputs."""
        activations = np.asarray(inputs, dtype=float)
        if activations.ndim == 1:
            activations = activations.reshape(1, -1)
        if activations.shape[1] != self.input_dim:
            raise ValueError("Input width does not match the network input dimension.")

        for weight, bias in zip(self.weights[:-1], self.biases[:-1]):
            activations = np.maximum(0.0, activations @ weight + bias)
        return activations @ self.weights[-1] + self.biases[-1]

    def backward(self, inputs, output_gradient, learning_rate):
        """Backpropagate an output gradient and update every network layer."""
        activations = np.asarray(inputs, dtype=float)
        if activations.ndim == 1:
            activations = activations.reshape(1, -1)

        layer_inputs = [activations]
        pre_activations = []
        for weight, bias in zip(self.weights[:-1], self.biases[:-1]):
            pre_activation = layer_inputs[-1] @ weight + bias
            pre_activations.append(pre_activation)
            layer_inputs.append(np.maximum(0.0, pre_activation))

        gradient = np.asarray(output_gradient, dtype=float)
        if gradient.ndim == 1:
            gradient = gradient.reshape(1, -1)
        if gradient.shape != (len(activations), self.output_dim):
            raise ValueError("Output-gradient shape does not match the network output.")

        # Save each layer's gradient before changing its weights for consistent backpropagation.
        weight_gradients = [None] * len(self.weights)
        bias_gradients = [None] * len(self.biases)
        for layer_index in range(len(self.weights) - 1, -1, -1):
            weight_gradients[layer_index] = layer_inputs[layer_index].T @ gradient
            bias_gradients[layer_index] = np.sum(gradient, axis=0)
            if layer_index > 0:
                gradient = gradient @ self.weights[layer_index].T
                gradient *= pre_activations[layer_index - 1] > 0.0

        for layer_index in range(len(self.weights)):
            self.weights[layer_index] -= learning_rate * weight_gradients[layer_index]
            self.biases[layer_index] -= learning_rate * bias_gradients[layer_index]


@dataclass
class Order:
    """A customer order with volume, location, priority, and deadline."""
    id: int
    volume: int
    location: Tuple[float, float]
    priority: int = 1
    deadline: int = 100


@dataclass
class AgentState:
    """Mutable operational statistics for one agent."""
    position: Tuple[float, float]
    current_batch: List[int]
    capacity_used: int
    total_distance: float
    orders_completed: int

print("Neural network and order data structures are ready.")

In [ ]:
class MultiAgentBatchingEnvironment:
    """Order-batching simulation with shared order availability and agent rewards."""

    def __init__(self, orders, num_agents=3, capacity=8, warehouse_size=(10, 15)):
        if not orders:
            raise ValueError("At least one order is required.")
        if num_agents <= 0 or capacity <= 0:
            raise ValueError("The number of agents and capacity must be positive.")
        if any(order.volume <= 0 for order in orders):
            raise ValueError("Every order must have positive volume.")

        self.orders = list(orders)
        self.num_orders = len(self.orders)
        self.num_agents = num_agents
        self.capacity = capacity
        self.warehouse_size = warehouse_size
        self.max_time_steps = 200
        self.agents = []
        self.distances = self._calculate_distances()
        self.coordination_bonus_history = []
        self.conflict_penalty_history = []
        self.reset()

    def _calculate_distances(self):
        """Build the Euclidean distance matrix, with index zero reserved for the depot."""
        distances = np.zeros((self.num_orders + 1, self.num_orders + 1))
        for index, order in enumerate(self.orders, start=1):
            x, y = order.location
            distances[0, index] = distances[index, 0] = np.hypot(x, y)

        for first_index, first_order in enumerate(self.orders, start=1):
            for second_index, second_order in enumerate(self.orders, start=1):
                if first_index != second_index:
                    distances[first_index, second_index] = np.hypot(
                        first_order.location[0] - second_order.location[0],
                        first_order.location[1] - second_order.location[1],
                    )
        return distances

    def _unassigned_available_orders(self):
        """Return available orders that are not already in any agent's open batch."""
        assigned = {
            order_id
            for agent in self.agents
            for order_id in agent.current_batch
        }
        return [
            order_id
            for order_id in self.available_orders
            if order_id not in assigned
        ]

    def get_state_for_agent(self, agent_id):
        """Encode candidate orders, this agent, its peers, and episode progress."""
        if not 0 <= agent_id < self.num_agents:
            raise IndexError("Agent index is outside the environment.")

        agent = self.agents[agent_id]
        candidate_orders = self._unassigned_available_orders()
        order_features = []
        for slot in range(5):
            if slot < len(candidate_orders):
                order_id = candidate_orders[slot]
                order = self.orders[order_id]
                remaining_deadline_fraction = max(
                    0, order.deadline - self.time_step
                ) / max(1, order.deadline)
                order_features.extend([
                    order.volume,
                    order.priority,
                    self.distances[0, order_id + 1],
                    remaining_deadline_fraction,
                ])
            else:
                order_features.extend([0, 0, 0, 0])

        agent_features = [
            agent.capacity_used / self.capacity,
            len(agent.current_batch),
            agent.total_distance / 100.0,
            agent.orders_completed / self.num_orders,
        ]

        other_agents_features = []
        for other_id, other_agent in enumerate(self.agents):
            if other_id == agent_id:
                other_agents_features.extend([0, 0])
            else:
                other_agents_features.extend([
                    other_agent.capacity_used / self.capacity,
                    len(other_agent.current_batch),
                ])

        environment_features = [
            len(self.available_orders) / self.num_orders,
            self.time_step / self.max_time_steps,
            len(self.completed_orders) / self.num_orders,
        ]
        state = (
            order_features
            + agent_features
            + other_agents_features
            + environment_features
        )
        return np.asarray(state, dtype=np.float32)

    def get_valid_actions(self, agent_id):
        """Return wait, feasible batch completion, and feasible order selections."""
        if not 0 <= agent_id < self.num_agents:
            raise IndexError("Agent index is outside the environment.")

        agent = self.agents[agent_id]
        valid_actions = [0]  # Action 0 means wait.
        if agent.current_batch:
            valid_actions.append(1)  # Action 1 means complete the current batch.

        for candidate_index, order_id in enumerate(self._unassigned_available_orders()):
            order = self.orders[order_id]
            if agent.capacity_used + order.volume <= self.capacity:
                valid_actions.append(2 + candidate_index)
        return valid_actions

    def step(self, actions):
        """Apply one action per agent and return rewards and the terminal flag."""
        if len(actions) != self.num_agents:
            raise ValueError("One action is required for each agent.")

        rewards = [0.0] * self.num_agents
        candidate_orders = self._unassigned_available_orders()

        for agent_id, action in enumerate(actions):
            agent = self.agents[agent_id]
            if action == 0:
                rewards[agent_id] -= 0.1
                continue

            if action == 1:
                if not agent.current_batch:
                    rewards[agent_id] -= 2.0
                    continue

                batch = list(agent.current_batch)
                batch_size = len(batch)
                batch_volume = agent.capacity_used
                rewards[agent_id] += 20.0 * batch_size / max(1, batch_volume)
                for order_id in batch:
                    if order_id in self.available_orders:
                        self.available_orders.remove(order_id)
                        self.completed_orders.append(order_id)
                agent.orders_completed += batch_size
                # The recorded distance is a depot round-trip proxy, not a route length.
                agent.total_distance += sum(
                    2.0 * self.distances[0, order_id + 1]
                    for order_id in batch
                )
                agent.current_batch = []
                agent.capacity_used = 0
                continue

            if action >= 2:
                candidate_index = action - 2
                if candidate_index >= len(candidate_orders):
                    rewards[agent_id] -= 3.0
                    continue

                order_id = candidate_orders[candidate_index]
                order = self.orders[order_id]
                already_assigned = any(
                    order_id in other_agent.current_batch
                    for other_id, other_agent in enumerate(self.agents)
                    if other_id != agent_id
                )
                if order_id not in self.available_orders or already_assigned:
                    rewards[agent_id] -= self._calculate_conflict_penalty(
                        agent_id, order_id
                    )
                    continue
                if agent.capacity_used + order.volume > self.capacity:
                    rewards[agent_id] -= 5.0
                    continue

                agent.current_batch.append(order_id)
                agent.capacity_used += order.volume
                distance_penalty = self.distances[0, order_id + 1] * 0.1
                rewards[agent_id] += 10.0 - distance_penalty
                rewards[agent_id] += self._calculate_coordination_bonus(agent_id)
                continue

            rewards[agent_id] -= 3.0

        self.time_step += 1
        done = (
            len(self.available_orders) == 0
            or self.time_step >= self.max_time_steps
        )
        if done:
            completion_rate = len(self.completed_orders) / self.num_orders
            rewards = [reward + 50.0 * completion_rate for reward in rewards]
        return rewards, done

    def _calculate_coordination_bonus(self, agent_id):
        """Reward an order selection when it does not overlap a peer's batch."""
        agent = self.agents[agent_id]
        bonus = 0.0
        for other_id, other_agent in enumerate(self.agents):
            if other_id != agent_id and not (
                set(agent.current_batch) & set(other_agent.current_batch)
            ):
                bonus += 0.1
        return bonus

    def _calculate_conflict_penalty(self, agent_id, order_id):
        """Penalize an attempted selection of an order held by another agent."""
        penalty = 0.0
        for other_id, other_agent in enumerate(self.agents):
            if other_id != agent_id and order_id in other_agent.current_batch:
                penalty += 2.0
        return penalty if penalty > 0 else 2.0

    def reset(self):
        """Start a fresh episode with all orders available and agents at the depot."""
        self.available_orders = list(range(self.num_orders))
        self.completed_orders = []
        self.time_step = 0
        self.agents = [
            AgentState(
                position=(0.0, 0.0),
                current_batch=[],
                capacity_used=0,
                total_distance=0.0,
                orders_completed=0,
            )
            for _ in range(self.num_agents)
        ]
        self.coordination_bonus_history = []
        self.conflict_penalty_history = []
        return [self.get_state_for_agent(i) for i in range(self.num_agents)]

print("Multi-agent batching environment is ready.")

In [ ]:
class ActorCriticAgent:
    """Actor-Critic learner with masked actions and a replay buffer."""

    def __init__(self, state_dim, action_dim, hidden_dims=None, lr=0.001):
        if hidden_dims is None:
            hidden_dims = [128, 64]
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.lr = lr
        self.gamma = 0.99
        self.actor = SimpleNeuralNetwork(state_dim, hidden_dims, action_dim)
        self.critic = SimpleNeuralNetwork(state_dim, hidden_dims, 1)
        self.memory = deque(maxlen=10000)
        self.training_steps = 0
        self.loss_history = []

    def select_action(self, state, valid_actions, epsilon=0.1):
        """Choose a valid action using epsilon exploration or the masked actor."""
        if not valid_actions:
            raise ValueError("At least one valid action is required.")
        if np.random.random() < epsilon:
            return int(np.random.choice(valid_actions))

        logits = self.actor.forward(state)[0]
        valid_logits = logits[valid_actions]
        shifted_logits = valid_logits - np.max(valid_logits)
        probabilities = np.exp(shifted_logits)
        probabilities /= np.sum(probabilities)
        return int(np.random.choice(valid_actions, p=probabilities))

    def store_experience(
        self, state, action, reward, next_state, done, valid_actions
    ):
        """Store a transition and its behavior-time action mask for replay."""
        action_mask = np.zeros(self.action_dim, dtype=bool)
        action_mask[valid_actions] = True
        self.memory.append((
            np.asarray(state, dtype=np.float32).copy(),
            int(action),
            float(reward),
            np.asarray(next_state, dtype=np.float32).copy(),
            bool(done),
            action_mask,
        ))

    def train(self, batch_size=32):
        """Update the critic by squared error and actor by the advantage signal."""
        if len(self.memory) < batch_size:
            return

        batch = random.sample(self.memory, batch_size)
        states, actions, rewards, next_states, dones, action_masks = zip(*batch)
        states = np.asarray(states, dtype=float)
        next_states = np.asarray(next_states, dtype=float)
        actions = np.asarray(actions, dtype=int)
        rewards = np.asarray(rewards, dtype=float)
        dones = np.asarray(dones, dtype=float)
        action_masks = np.asarray(action_masks, dtype=bool)

        current_values = self.critic.forward(states).reshape(-1)
        next_values = self.critic.forward(next_states).reshape(-1)
        targets = rewards + self.gamma * next_values * (1.0 - dones)
        advantages = targets - current_values

        critic_gradient = (2.0 / batch_size) * (current_values - targets)
        self._update_network(self.critic, states, critic_gradient.reshape(-1, 1))
        self._update_policy(states, actions, advantages, action_masks)

        critic_loss = float(np.mean((targets - current_values) ** 2))
        self.loss_history.append(critic_loss)
        self.training_steps += 1

    def _update_network(self, network, inputs, output_gradient):
        """Apply a batch gradient to the value network through all its layers."""
        network.backward(inputs, output_gradient, self.lr)

    def _update_policy(self, states, actions, advantages, action_masks):
        """Apply the masked categorical policy-gradient update to the actor."""
        logits = self.actor.forward(states)
        masked_logits = np.where(action_masks, logits, -np.inf)
        row_maxima = np.max(masked_logits, axis=1, keepdims=True)
        probabilities = np.exp(masked_logits - row_maxima)
        probabilities /= np.sum(probabilities, axis=1, keepdims=True)

        logit_gradient = probabilities
        logit_gradient[np.arange(len(actions)), actions] -= 1.0
        logit_gradient *= advantages[:, None] / len(actions)
        self.actor.backward(states, logit_gradient, self.lr)

print("Actor-Critic agents are ready.")

In [ ]:
def create_example_orders(num_orders=50):
    """Generate the example order set with reproducible characteristics."""
    if num_orders <= 0:
        raise ValueError("The number of orders must be positive.")

    orders = []
    np.random.seed(42)
    for order_id in range(num_orders):
        volume = int(np.random.randint(1, 5))
        location = (
            float(np.random.uniform(0, 10)),
            float(np.random.uniform(0, 15)),
        )

        if order_id < num_orders // 3:
            priority = 1
            deadline = int(np.random.randint(80, 120))
        elif order_id < 2 * num_orders // 3:
            priority = 2
            deadline = int(np.random.randint(40, 80))
        else:
            priority = 3
            deadline = int(np.random.randint(20, 60))

        orders.append(Order(order_id, volume, location, priority, deadline))
    return orders


orders_example = create_example_orders(50)
env = MultiAgentBatchingEnvironment(
    orders_example,
    num_agents=3,
    capacity=8,
)

print("=== Problem instance ===")
print(f"Number of orders: {len(orders_example)}")
print(f"Number of agents: {env.num_agents}")
print(f"Capacity per agent: {env.capacity}")
print(f"Warehouse dimensions used for the example: {env.warehouse_size}")

total_volume = sum(order.volume for order in orders_example)
priority_distribution = defaultdict(int)
for order in orders_example:
    priority_distribution[order.priority] += 1

print("\nOrder characteristics:")
print(f"  Total order volume: {total_volume}")
print(f"  Mean volume per order: {total_volume / len(orders_example):.2f}")
print(
    "  Capacity-based lower bound on the number of batches: "
    f"{int(np.ceil(total_volume / env.capacity))}"
)
print(f"  Priority counts: {dict(priority_distribution)}")

sample_state = env.get_state_for_agent(0)
sample_actions = env.get_valid_actions(0)
print("\nState and action representation:")
print(f"  State dimension: {len(sample_state)}")
print(f"  First ten state features: {sample_state[:10].round(3)}")
print(f"  Valid actions for agent 0: {sample_actions}")
print(f"  Fixed action dimension: {len(orders_example) + 2}")

In [ ]:
def initialize_agents(env):
    """Create one Actor-Critic learner for each agent in the environment."""
    state_dim = len(env.get_state_for_agent(0))
    action_dim = env.num_orders + 2
    return [
        ActorCriticAgent(
            state_dim=state_dim,
            action_dim=action_dim,
            hidden_dims=[128, 64],
            lr=0.001,
        )
        for _ in range(env.num_agents)
    ]


agents = initialize_agents(env)
print("=== Multi-agent system initialization ===")
print(f"Number of agents: {len(agents)}")
print(f"State dimension per agent: {agents[0].state_dim}")
print(f"Action dimension per agent: {agents[0].action_dim}")
print(f"Hidden layers: {agents[0].actor.hidden_dims}")
print(f"Learning rate: {agents[0].lr}")
print(f"Replay capacity per agent: {agents[0].memory.maxlen} transitions")
print("Each actor represents a policy; each critic estimates state value.")

In [ ]:
def train_marl_system(env, agents, episodes=15000, max_steps_per_episode=200):
    """Train agents through shared-environment interaction and replay updates."""
    if episodes <= 0 or max_steps_per_episode <= 0:
        raise ValueError("Episode and step limits must be positive.")

    print("=== Training the multi-agent reinforcement-learning system ===")
    print(f"Episodes: {episodes}; maximum steps per episode: {max_steps_per_episode}")
    episode_rewards = []
    episode_completion_rates = []
    episode_lengths = []
    epsilon_start = 1.0
    epsilon_end = 0.01
    epsilon_decay = (epsilon_start - epsilon_end) / episodes
    start_time = time.time()

    for episode in range(episodes):
        states = env.reset()
        episode_reward = 0.0
        epsilon = max(epsilon_end, epsilon_start - episode * epsilon_decay)

        for step in range(max_steps_per_episode):
            valid_actions_by_agent = [
                env.get_valid_actions(agent_id)
                for agent_id in range(env.num_agents)
            ]
            actions = [
                agents[agent_id].select_action(
                    states[agent_id],
                    valid_actions_by_agent[agent_id],
                    epsilon,
                )
                for agent_id in range(env.num_agents)
            ]

            rewards, done = env.step(actions)
            next_states = [
                env.get_state_for_agent(agent_id)
                for agent_id in range(env.num_agents)
            ]
            for agent_id in range(env.num_agents):
                agents[agent_id].store_experience(
                    states[agent_id],
                    actions[agent_id],
                    rewards[agent_id],
                    next_states[agent_id],
                    done,
                    valid_actions_by_agent[agent_id],
                )

            episode_reward += sum(rewards)
            states = next_states
            if done:
                break

        # Delaying updates gives the replay buffers time to collect transitions.
        if episode > 100:
            for agent in agents:
                agent.train(batch_size=32)

        completion_rate = len(env.completed_orders) / len(env.orders)
        episode_rewards.append(episode_reward)
        episode_completion_rates.append(completion_rate)
        episode_lengths.append(step + 1)

        if episode % 1000 == 0:
            recent_start = max(0, len(episode_rewards) - 1000)
            average_reward = np.mean(episode_rewards[recent_start:])
            average_completion = np.mean(
                episode_completion_rates[recent_start:]
            )
            average_length = np.mean(episode_lengths[recent_start:])
            elapsed_time = time.time() - start_time
            print(
                f"Episode {episode:5d}: reward={average_reward:8.2f}, "
                f"completion={average_completion:5.3f}, "
                f"length={average_length:5.1f}, epsilon={epsilon:.3f}, "
                f"elapsed={elapsed_time:.1f}s"
            )

    total_time = time.time() - start_time
    print("\nTraining run completed.")
    print(f"Elapsed training time: {total_time:.2f} seconds")
    print(f"Mean reward over final episodes: {np.mean(episode_rewards[-1000:]):.2f}")
    print(
        "Mean completion rate over final episodes: "
        f"{np.mean(episode_completion_rates[-1000:]):.3f}"
    )
    return {
        "episode_rewards": episode_rewards,
        "episode_completion_rates": episode_completion_rates,
        "episode_lengths": episode_lengths,
        "training_time": total_time,
    }


training_results = train_marl_system(env, agents, episodes=15000)

In [ ]:
def evaluate_trained_agents(env, agents, num_test_episodes=100):
    """Measure deterministic-policy performance and completed-batch patterns."""
    if num_test_episodes <= 0:
        raise ValueError("The number of evaluation episodes must be positive.")

    print("=== Evaluating trained agents ===")
    test_rewards = []
    test_completion_rates = []
    specializations = []

    for _ in range(num_test_episodes):
        states = env.reset()
        episode_reward = 0.0
        completed_batch_sizes = [[] for _ in range(env.num_agents)]
        completed_batch_volumes = [[] for _ in range(env.num_agents)]

        for _step in range(env.max_time_steps):
            actions = [
                agents[agent_id].select_action(
                    states[agent_id],
                    env.get_valid_actions(agent_id),
                    epsilon=0.0,
                )
                for agent_id in range(env.num_agents)
            ]
            batches_before_step = [
                list(agent.current_batch) for agent in env.agents
            ]
            rewards, done = env.step(actions)
            next_states = [
                env.get_state_for_agent(agent_id)
                for agent_id in range(env.num_agents)
            ]

            for agent_id, action in enumerate(actions):
                if action == 1 and batches_before_step[agent_id]:
                    batch = batches_before_step[agent_id]
                    completed_batch_sizes[agent_id].append(len(batch))
                    completed_batch_volumes[agent_id].append(
                        sum(env.orders[order_id].volume for order_id in batch)
                    )

            episode_reward += sum(rewards)
            states = next_states
            if done:
                break

        test_rewards.append(episode_reward)
        test_completion_rates.append(
            len(env.completed_orders) / len(env.orders)
        )
        for agent_id in range(env.num_agents):
            if completed_batch_sizes[agent_id]:
                specializations.append({
                    "agent_id": agent_id,
                    "avg_batch_size": float(
                        np.mean(completed_batch_sizes[agent_id])
                    ),
                    "avg_batch_volume": float(
                        np.mean(completed_batch_volumes[agent_id])
                    ),
                    "num_batches": len(completed_batch_sizes[agent_id]),
                })

    specialization_df = pd.DataFrame(specializations)
    specialization_summary = None
    if not specialization_df.empty:
        specialization_summary = specialization_df.groupby("agent_id").agg({
            "avg_batch_size": "mean",
            "avg_batch_volume": "mean",
            "num_batches": "sum",
        }).round(2)

    average_reward = float(np.mean(test_rewards))
    average_completion = float(np.mean(test_completion_rates))
    print("\nEvaluation results:")
    print(f"  Mean episode reward: {average_reward:.2f}")
    print(f"  Mean order completion rate: {average_completion:.3f}")
    print(f"  Reward standard deviation: {np.std(test_rewards):.2f}")
    print(f"  Completion-rate standard deviation: {np.std(test_completion_rates):.3f}")

    if specialization_summary is not None:
        print("\nMean completed-batch statistics by agent:")
        print(specialization_summary)
        for agent_id, row in specialization_summary.iterrows():
            if row["avg_batch_size"] > 6:
                pattern = "larger-batch pattern"
            elif row["avg_batch_size"] < 4:
                pattern = "smaller-batch pattern"
            else:
                pattern = "intermediate-batch pattern"
            print(
                f"  Agent {agent_id}: {pattern}; "
                f"mean batch size={row['avg_batch_size']:.1f}, "
                f"mean volume={row['avg_batch_volume']:.1f}"
            )
    else:
        print("No completed batches were recorded for specialization analysis.")

    return {
        "avg_reward": average_reward,
        "avg_completion": average_completion,
        "specialization_summary": specialization_summary,
    }


evaluation_results = evaluate_trained_agents(
    env,
    agents,
    num_test_episodes=100,
)

In [ ]:
def train_independent_agents(env, episodes=5000):
    """Train the same agents with coordination reward terms disabled."""
    print("=== Training the independent-agent baseline ===")
    independent_agents = initialize_agents(env)
    original_coordination_bonus = env._calculate_coordination_bonus
    original_conflict_penalty = env._calculate_conflict_penalty

    # The baseline keeps the environment and action process while removing these two reward terms.
    env._calculate_coordination_bonus = lambda agent_id: 0.0
    env._calculate_conflict_penalty = lambda agent_id, order_id: 0.0
    try:
        independent_results = train_marl_system(
            env,
            independent_agents,
            episodes=episodes,
        )
    finally:
        env._calculate_coordination_bonus = original_coordination_bonus
        env._calculate_conflict_penalty = original_conflict_penalty

    independent_evaluation = evaluate_trained_agents(
        env,
        independent_agents,
        num_test_episodes=50,
    )
    return independent_results, independent_evaluation


independent_results, independent_evaluation = train_independent_agents(
    env,
    episodes=5000,
)

baseline_reward = independent_evaluation["avg_reward"]
if not np.isclose(baseline_reward, 0.0):
    reward_improvement = (
        (evaluation_results["avg_reward"] - baseline_reward)
        / abs(baseline_reward)
        * 100.0
    )
else:
    reward_improvement = np.nan

baseline_completion = independent_evaluation["avg_completion"]
if not np.isclose(baseline_completion, 0.0):
    completion_improvement = (
        (evaluation_results["avg_completion"] - baseline_completion)
        / abs(baseline_completion)
        * 100.0
    )
else:
    completion_improvement = np.nan

print("\nCoordination comparison:")
print(f"  Independent-agent mean reward: {baseline_reward:.2f}")
print(f"  Coordinated-agent mean reward: {evaluation_results['avg_reward']:.2f}")
print(f"  Relative reward change: {reward_improvement:.1f}%")
print(f"  Independent-agent completion: {baseline_completion:.3f}")
print(f"  Coordinated-agent completion: {evaluation_results['avg_completion']:.3f}")
print(f"  Relative completion change: {completion_improvement:.1f}%")

In [ ]:
def create_visualizations(
    training_results,
    evaluation_results,
    independent_evaluation,
    reward_improvement,
    completion_improvement,
):
    """Plot learning history, batch statistics, and baseline comparisons."""
    figure, axes = plt.subplots(2, 2, figsize=(15, 12))
    figure.suptitle(
        "Order Batching: Multi-Agent Reinforcement-Learning Analysis",
        fontsize=16,
        fontweight="bold",
    )

    # Episode-level curves are smoothed for readability; raw results remain available.
    episode_count = len(training_results["episode_rewards"])
    episodes = np.arange(episode_count)
    window = min(100, max(1, episode_count))
    smoothed_rewards = pd.Series(
        training_results["episode_rewards"]
    ).rolling(window, min_periods=1).mean()
    smoothed_completion = pd.Series(
        training_results["episode_completion_rates"]
    ).rolling(window, min_periods=1).mean()

    reward_axis = axes[0, 0]
    reward_line, = reward_axis.plot(
        episodes,
        smoothed_rewards,
        color="blue",
        label="Smoothed episode reward",
        linewidth=2,
    )
    reward_axis.set_xlabel("Episode")
    reward_axis.set_ylabel("Reward", color="blue")
    reward_axis.tick_params(axis="y", labelcolor="blue")
    reward_axis.set_title("Training history")
    reward_axis.grid(True, alpha=0.3)
    completion_axis = reward_axis.twinx()
    completion_line, = completion_axis.plot(
        episodes,
        smoothed_completion,
        color="red",
        linestyle="--",
        label="Smoothed completion rate",
        linewidth=2,
    )
    completion_axis.set_ylabel("Completion rate", color="red")
    completion_axis.tick_params(axis="y", labelcolor="red")
    reward_axis.legend(
        [reward_line, completion_line],
        [reward_line.get_label(), completion_line.get_label()],
        loc="best",
    )

    specialization_axis = axes[0, 1]
    summary = evaluation_results["specialization_summary"]
    if summary is not None:
        positions = np.arange(len(summary))
        width = 0.35
        specialization_axis.bar(
            positions - width / 2,
            summary["avg_batch_size"],
            width,
            label="Mean orders per completed batch",
            color="skyblue",
        )
        specialization_axis.bar(
            positions + width / 2,
            summary["avg_batch_volume"],
            width,
            label="Mean volume per completed batch",
            color="lightcoral",
        )
        specialization_axis.set_xticks(positions)
        specialization_axis.set_xticklabels(
            [f"Agent {agent_id}" for agent_id in summary.index]
        )
        specialization_axis.set_xlabel("Agent")
        specialization_axis.set_ylabel("Mean batch measure")
        specialization_axis.legend()
        specialization_axis.grid(True, axis="y", alpha=0.3)
    else:
        specialization_axis.text(
            0.5,
            0.5,
            "No completed-batch data available",
            ha="center",
            va="center",
            transform=specialization_axis.transAxes,
        )
    specialization_axis.set_title("Completed-batch statistics")

    comparison_axis = axes[1, 0]
    methods = ["Independent agents", "Coordinated agents"]
    rewards = [
        independent_evaluation["avg_reward"],
        evaluation_results["avg_reward"],
    ]
    completion_rates = [
        independent_evaluation["avg_completion"],
        evaluation_results["avg_completion"],
    ]
    positions = np.arange(len(methods))
    width = 0.35
    comparison_axis.bar(
        positions - width / 2,
        rewards,
        width,
        label="Mean episode reward",
        color="lightgreen",
    )
    completion_axis = comparison_axis.twinx()
    completion_axis.bar(
        positions + width / 2,
        completion_rates,
        width,
        label="Mean completion rate",
        color="orange",
    )
    comparison_axis.set_xticks(positions)
    comparison_axis.set_xticklabels(methods)
    comparison_axis.set_xlabel("Training setup")
    comparison_axis.set_ylabel("Mean episode reward")
    completion_axis.set_ylabel("Mean completion rate")
    comparison_axis.set_title("Evaluation comparison")
    comparison_axis.grid(True, axis="y", alpha=0.3)
    comparison_axis.legend(loc="upper left")
    completion_axis.legend(loc="upper right")
    comparison_axis.text(
        0.02,
        0.95,
        f"Reward change: {reward_improvement:.1f}%",
        transform=comparison_axis.transAxes,
        va="top",
    )
    comparison_axis.text(
        0.02,
        0.87,
        f"Completion change: {completion_improvement:.1f}%",
        transform=comparison_axis.transAxes,
        va="top",
    )

    length_axis = axes[1, 1]
    episode_lengths = training_results["episode_lengths"]
    length_axis.hist(
        episode_lengths,
        bins=30,
        color="purple",
        alpha=0.7,
        edgecolor="black",
    )
    mean_length = float(np.mean(episode_lengths))
    length_axis.axvline(
        mean_length,
        color="red",
        linestyle="--",
        linewidth=2,
        label=f"Mean: {mean_length:.1f} steps",
    )
    length_axis.set_xlabel("Steps per episode")
    length_axis.set_ylabel("Episode count")
    length_axis.set_title("Training episode lengths")
    length_axis.legend()
    length_axis.grid(True, alpha=0.3)

    figure.tight_layout()
    plt.show()
    return figure


visualization_fig = create_visualizations(
    training_results,
    evaluation_results,
    independent_evaluation,
    reward_improvement,
    completion_improvement,
)
print("Evaluation visualizations are ready.")

In [ ]:
print("=== Multi-agent reinforcement-learning analysis ===")
print("\nSystem characteristics:")
print(f"  Agents: {env.num_agents}")
print(f"  State features per agent: {agents[0].state_dim}")
print(f"  Fixed action dimension per agent: {agents[0].action_dim}")
print(f"  Actor hidden-layer widths: {agents[0].actor.hidden_dims}")
print(f"  Replay-buffer capacity per agent: {agents[0].memory.maxlen}")

print("\nComputational considerations:")
print(
    "  Training work grows with episode count, steps per episode, "
    "number of agents, and neural-network update cost."
)
print(
    "  Replay storage grows with buffer capacity and the size of each "
    "state-action transition."
)
print(
    "  The reported neural-network work is performed by NumPy operations; "
    "the networks are not a deep-learning framework implementation."
)

print("\nCoordination mechanisms represented in the environment:")
print("  Agent observations include simplified information about peer batches.")
print("  Coordination rewards favor selections that do not overlap peer batches.")
print("  Duplicate selections are rejected and receive a conflict penalty.")
print("  Policies are decentralized, while agents interact through one shared environment.")

print("\nLearning and interpretation notes:")
print("  Actors use masked action probabilities and epsilon-greedy exploration.")
print("  Critics estimate discounted state value using one-step TD targets.")
print("  Replay updates reuse stored transitions; no off-policy correction is applied.")
print("  Evaluation metrics describe this simulation and do not establish a general guarantee.")
print("  Priority and deadline affect observations but have no separate reward term.")
print("  Physical picker routing, travel time, congestion, and shift scheduling are outside scope.")

print("\nObserved training and evaluation metrics:")
print(f"  Training episodes: {len(training_results['episode_rewards'])}")
print(f"  Training elapsed time: {training_results['training_time']:.2f} seconds")
print(f"  Coordinated evaluation reward: {evaluation_results['avg_reward']:.2f}")
print(f"  Coordinated completion rate: {evaluation_results['avg_completion']:.3f}")
print(f"  Relative reward change versus baseline: {reward_improvement:.1f}%")

In [ ]:
print("=== Summary and conclusions ===")
print("\nMetrics from this run:")
print(f"  Relative reward change versus the independent-agent baseline: {reward_improvement:.1f}%")
print(f"  Coordinated evaluation completion rate: {evaluation_results['avg_completion']:.1%}")
print(f"  Training episodes: {len(training_results['episode_rewards'])}")
print(f"  Mean training episode length: {np.mean(training_results['episode_lengths']):.1f} steps")
print(f"  Training time: {training_results['training_time']:.2f} seconds")

print("\nWhat the example demonstrates:")
print("  Multiple agents can learn separate policies while interacting through shared order availability.")
print("  Actor-Critic learning combines a policy model with a learned state-value estimate.")
print("  Experience replay reuses transitions for stochastic gradient updates.")
print("  Reward shaping provides explicit signals for order selection and coordination.")
print("  Evaluation can summarize completion, reward, and completed-batch patterns.")

print("\nHow to interpret the comparison:")
print("  The baseline removes coordination bonus and conflict penalty terms while retaining the environment.")
print("  Reward and completion changes are empirical comparisons for these training runs and this instance.")
print("  A positive change in one metric does not guarantee improvement in every operational objective.")
print("  The simulation does not establish performance on larger or real warehouse data.")

print("\nNatural extensions within this modeling approach:")
print("  Additional operational constraints can be represented in the environment state and valid actions.")
print("  More realistic travel, congestion, and deadline effects require corresponding transition and reward models.")
print("  Multiple random seeds and larger test instances can help assess variability and generalization.")